In [447]:
# Board Square = color: 1 bit, piece_type: 3 bits

class Piece:
    WHITE_VALUE = 0b0000
    BLACK_VALUE = 0b1000
    
    @staticmethod
    def isWhite(coloredPieceValue):
        return Piece.extractColor(coloredPieceValue) == 0
    
    @staticmethod
    def extractColor(coloredPieceValue):
        return (coloredPieceValue & 0b1000)

    @staticmethod
    def extractPiece(coloredPieceValue):
        return (coloredPieceValue & 0b0111)

    def __init__(self, name, symbols, value, moveDefinitions, validMovesFunc):
        self.name = name
        self.symbols = symbols
        self.value = value
        self.moveDefinitions = moveDefinitions
        self.validMovesFunc = validMovesFunc
    def __str__(self):
        return self.name

    def getValidMoves(self, state, position):
        return self.validMovesFunc(state, position, self.moveDefinitions, Piece.isWhite(state.getAt(position)))
    def getSymbol(self, asWhite):
        return self.symbols[0 if asWhite else 1]
    def getValue(self, asWhite):
        return self.value | (Piece.WHITE_VALUE if asWhite else Piece.BLACK_VALUE)
    def isValue(self, value):
        return (value & 0b0111) == self.value
    

class MoveDef:
    def __init__(self, direction, strength = -1, mirrorX = False, mirrorY = False, jump = False, canCapture = False, mustCapture = False):
        self.boundsX = 8
        self.boundsY = 8
        
        self.direction = direction
        self.strength = strength
        self.mirrorX = mirrorX
        self.mirrorY = mirrorY
        self.jump = jump
        self.canCapture = canCapture
        self.mustCapture = mustCapture
    def getPossibleFrom(self, x, y, flipY = False):
        possible = []

        if(not self.jump):
            directions = [(self.direction[0], self.direction[1] if not flipY else -1 * self.direction[1])]
            root_dir = directions[0]
            if(self.mirrorX):
                directions.append((root_dir[0] * -1, root_dir[1]))
            if(self.mirrorY):
                directions.append((root_dir[0], root_dir[1] * - 1))

            for direction in directions:
                strengthI = self.strength
                curX = x
                curY = y
                for i in range(strengthI):
                    curX += direction[0] + i
                    curY += direction[1] + i
                    if(curX < 0 or curX >= self.boundsX or curY < 0 or curY >= self.boundsY):
                        break
                    possible.append((curX, curY))
        elif(self.jump):
            directions = [(self.direction[0], self.direction[1] if not flipY else -1 * self.direction[1])]
            if(self.mirrorX):
                directions.append((directions[0] * -1, directions[1]))
            if(self.mirrorY):
                directions.append((directions[0], directions[1] * - 1))

            for direction in directions:
                possible.append(x + directions[0], y + directions[1])

        return possible

def pawnValidMoves(state, position, moveDefinitions, isWhite):
    x, y = state.getXY(position)

    if(x == -1 or y == -1):
        raise Exception("Attempted to solve moves for piece at invalid position index " + position + ".")

    states = []

    for move in moveDefinitions:
        possibles = move.getPossibleFrom(x, y, not isWhite)
        
        for position in possibles:
            value = state.getAt2D(position[0], position[1])

            if(
                PieceDefs.isEmpty(value) and not move.mustCapture or
                (not PieceDefs.isEmpty(value) and move.canCapture and Piece.extractColor(value) != Piece.WHITE_VALUE if isWhite else Piece.BLACK_VALUE )):
                states.append(state.copyWithMove(x, y, position[0], position[1]))

    return states

class PieceDefs:
    EMPTY_VALUE = 0b000
    PAWN_VALUE = 0b001
    KNIGHT_VALUE = 0b010
    BISHOP_VALUE = 0b011
    ROOK_VALUE = 0b100
    QUEEN_VALUE = 0b101
    KING_VALUE = 0b110

    EMPTY = Piece("", ("  ", "  "), 0b000, [], lambda state, position, moveDefinitions, isWhite: [])
    PAWN = Piece("Pawn", ("P", "p"), 0b001, [
        MoveDef((0,1), 1, False, False, False, False, False),
        MoveDef((1,1), 1, True, False, False, True, True),
    ], pawnValidMoves)
    
    VALUE_MAP = {
        EMPTY_VALUE: EMPTY,
        PAWN_VALUE: PAWN
    }

    @staticmethod
    def isEmpty(value):
        return value == PieceDefs.EMPTY.getValue(True) or value == PieceDefs.EMPTY.getValue(False)

class BoardState:
    def __init__(self, state = 0):
        self.width = 8
        self.height = 8

        self.state = state
    
    def inBounds(self, x, y):
        return x >= 0 and x < self.width and y >= 0 and y < self.height

    def copyWithMove(self, x, y, newX, newY):
        copy = BoardState(self.state)
        val = copy.getAt2D(x, y)
        copy.setAt2D(x, y, PieceDefs.EMPTY.getValue(True))
        copy.setAt2D(newX, newY, val)
        return copy

    def getXY(self, i):
        x = i % self.width
        y = self.height - 1 - (i // self.width)

        if(self.inBounds(x, y)):
            return x, y
        else:
            return -1, -1

    def getI(self, x, y):
        if(self.inBounds(x, y)):
            return (self.height - 1 - y) * self.width + x
        else:
            return -1

    def getAt2D(self, x, y):
        i = self.getI(x, y)
        if(i == -1):
            return None

        return self.getAt(i)

    def setAt2D(self, x, y, pieceValue):
        i = self.getI(x, y)
        if(i == -1):
            return

        return self.setAt(i, pieceValue)

    def getAt(self, i):
        return (self.state >> (i * 4)) & 0b1111
    
    def setAt(self, i, pieceValue):
        mask = 0b1111 << i * 4
        self.state &= ~mask
        self.state |= pieceValue << i * 4

    def __str__(self):
        box_width = 3

        string = ""
        string += " " + " ".center(box_width) + " "
        string += "┌"
        for x in range(self.width - 1):
            string += "─" * box_width + "┬"
        string += "─" * box_width + "┐\n"

        for y in range(self.height):

            string += " "
            string += str(self.width - y).center(box_width) + " "

            string += "│"
            for x in range(self.width):
                pieceValue = self.getAt2D(x, self.height - y - 1)
                piece = PieceDefs.VALUE_MAP[Piece.extractPiece(pieceValue)]
                symbol = piece.getSymbol(Piece.isWhite(pieceValue))
                string += symbol.center(box_width) + "│"

            string += "\n " + " ".center(box_width) + " "

            if(y < self.height - 1):
                string += "├" + (("─" * box_width + "┼") * (self.width - 1)) + ("─" * box_width + "┤")
            else:
                string += "└" + (("─" * box_width + "┴") * (self.width - 1)) + ("─" * box_width + "┘")

            string += "\n"
        
        string += "  " + " ".center(box_width) + " "
        for letter in ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']:
            string += letter.center(box_width) + " "

        return string

In [448]:
state = BoardState()

state.setAt2D(1,1, PieceDefs.PAWN.getValue(True))
state.setAt2D(0,2, PieceDefs.PAWN.getValue(False))
state.setAt2D(2,2, PieceDefs.PAWN.getValue(True))

PieceDefs.PAWN.getSymbol(True)

print(state)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │ p │   │ P │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │ P │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  


In [449]:
states = PieceDefs.PAWN.getValidMoves(state, state.getI(1, 1))

for state in states:
    print(state)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │ p │ P │ P │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   